# Face Recognition Project

## 0. Introduction

## 1. Get Setup

### 1.1 Download Dataset Helper Function

In [5]:
import os
import zipfile
from pathlib import Path
import requests

def download_data(source: str, 
                    destination: str,
                    remove_source: bool = True) -> Path:
    """Downloads a zipped dataset from source and unzips to destination.

    Args:
        source (str): A link to a zipped file containing data.
        destination (str): A target directory to unzip data to.
        remove_source (bool): Whether to remove the source after downloading and extracting.
    
    Returns:
        pathlib.Path to downloaded data.
    
    Example usage:
        download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip",
        destination="pizza_steak_sushi")
    """
    # Setup path to data folder
    data_path = Path("data/")
    image_path = data_path / destination

    # If the image folder doesn't exist, download it and prepare it... 
    if image_path.is_dir():
        print(f"[INFO] {image_path} directory exists, skipping download.")
    else:
        print(f"[INFO] Did not find {image_path} directory, creating one...")
        image_path.mkdir(parents=True, exist_ok=True)
        
        # Download pizza, steak, sushi data
        target_file = Path(source).name
        with open(data_path / target_file, "wb") as f:
            request = requests.get(source)
            print(f"[INFO] Downloading {target_file} from {source}...")
            f.write(request.content)

        # Unzip pizza, steak, sushi data
        with zipfile.ZipFile(data_path / target_file, "r") as zip_ref:
            print(f"[INFO] Unzipping {target_file} data...") 
            zip_ref.extractall(image_path)

        # Remove .zip file
        if remove_source:
            os.remove(data_path / target_file)
    
    return image_path

### 1.2 Unzip Dataset

In [7]:
try:
    print("[INFO] Unzipping dataset...")
    !unzip dataset.zip
except:
    print("[INFO] Could not unzip dataset... downloading and unzipping dataset...")
    download_data(source="https://github.com/Axeloooo/Face-Recognition/raw/devel/data/dataset.zip",
                    destination="dataset")
    !unzip dataset.zip

[INFO] Unzipping dataset...
unzip:  cannot find or open dataset.zip, dataset.zip.zip or dataset.zip.ZIP.


## 2. Get Data

In [1]:
import cv2
import numpy as np
from glob import glob
# custom load data function to load images and record subject labels
def get_data(path: str):
    paths = glob(path, recursive=True)
    data = [] #list of images
    label = [] #list of labels
    for path in paths:
        img = cv2.imread(path,0) # read image
        subject_label = path[path.rfind("s")+1:path.rfind("\\")] #extract the label
        # pre−processing step
        # can resize, rescale, normalize
        img = cv2.resize(img,(1,len(img)*len(img[0]))) # reshape image to a 1D vector
        img = np.float32(np.array(img)/255.0) #normalize to 0−1 value
        # can apply LBP, PCA or other forms of feature extraction
        # append images and labels
        data.append(img)
        # decrease all labels by 1 since subject labels start from 1
        label.append(int(subject_label)-1)
    return np.array(data)[:,:,0], np.array(label)

## 3. Local Binary Pattern (LBP) 

In [4]:
from skimage.feature import local_binary_pattern
from skimage.io import imread
# Read one image of subject 1 from dataset
img = imread("ATT dataset/s1/1.pgm", as_gray=True)
# Extract LBP feature from the image
# P: Number of circularly symmetric neighbor set points = 12
# Q: Radius of circle = 3
lbp = local_binary_pattern(img, 12, 3)

## 3. Support Vector Machines (SVM)

In [ ]:
from sklearn.svm import SVC
import numpy as np
train_path = ""  
test_path = ""
train_data , train_label = get_data(train_path) # use the previous custom get data function
test_data , test_label = get_data(test_path) # use the previous custom get data function
svm = SVC(C=5.0, gamma=0.001, probability=True) #experiment with different C and gamma
svm.fit(train_data , train_label)
# probability matrix NxM where N is number of samples and M is the number of classes
probability_matrix = svm.predict_proba(test_data)
# calculate accuracy
prediction = np.argmax(probability_matrix ,1)
result = prediction == test_label
accuracy = np.sum(result)/len(result)

## 4. Multi-Layer Perceptron (MLP)

In [ ]:
from sklearn.neural_network import MLPClassifier
import numpy as np
train_path = ""
test_path = ""
# customize get data function to include pre−processing methods (adding PCA or LBP)
train_data , train_label = get_data(train_path) # use the previous custom get data function
test_data , test_label = get_data(test_path) # use the previous custom get data function
# create MLP with 3 layers of perceptrons
# first layers has 128 neurons then 64 then another 128
# experiment with different layers/neurons
# experiment with different learning rate
mlp = MLPClassifier(hidden_layer_sizes=(128,64,128),
                    learning_rate_init=0.001,
                    random_state=1)
mlp.fit(train_data , train_label)
# probability matrix NxM where N is number of samples and M is the number of classes
probability_matrix = mlp.predict_proba(test_data)
# calculate accuracy
prediction = np.argmax(probability_matrix ,1)
result = prediction == test_label
accuracy = np.sum(result)/len(result)

## 5. ROC (FPR vs. TPR)

## 6. DET (FPR vs. FNR)

## 7. Conclusion